In [ ]:
%reload_ext autoreload
%autoreload 2
%matplotlib inline
%autosave 300

In [ ]:
import os

os.chdir("../")
print(os.getcwd())

## In this module we will implement logistic regression from scratch using PyTorch based on toy dataset, We will implement a basic classifier without using base pytorch and also data loaders.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch

In [ ]:
df = pd.read_csv("data/perceptron_toydata-truncated.txt", sep="\t")
df.head()

In [ ]:
X_train = df[["x1", "x2"]].values
y_train = df["label"].values

In [ ]:
# Data normalization
def normalize_data(X):
    """Normalize the data to [0, 1] range."""
    X_min = X.min(axis=0)
    X_max = X.max(axis=0)
    X_norm = (X - X_min) / (X_max - X_min)
    return X_norm


def standardize_data(X):
    """Standardize the data to have mean 0 and std 1."""
    X_mean = X.mean(axis=0)
    X_std = X.std(axis=0)
    X_stdized = (X - X_mean) / X_std
    return X_stdized

In [ ]:
X_train = normalize_data(X_train)

In [ ]:
X_train.shape, y_train.shape

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

In [ ]:
plt.plot(
    X_train[y_train == 0, 0],
    X_train[y_train == 0, 1],
    marker="D",
    markersize=10,
    linestyle="",
    label="Class 0",
)

plt.plot(
    X_train[y_train == 1, 0],
    X_train[y_train == 1, 1],
    marker="^",
    markersize=13,
    linestyle="",
    label="Class 1",
)

plt.legend(loc=2)

plt.xlim([-5, 5])
plt.ylim([-5, 5])

plt.xlabel("Feature $x_1$", fontsize=12)
plt.ylabel("Feature $x_2$", fontsize=12)

plt.grid()
plt.show()

#### Dataloaders and Datasets in PyTorch

In [ ]:
class CustomDataset(Dataset):
    """Custom Dataset for loading data."""

    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)
        self.n_samples = X.shape[0]

    def __getitem__(self, index):
        return self.X[index], self.y[index]

    def __len__(self):
        return self.n_samples

In [ ]:
train_ds = CustomDataset(X_train, y_train)

train_loader = DataLoader(
    dataset=train_ds,
    batch_size=10,
    shuffle=True,
)

In [ ]:
# for X_batch, y_batch in train_loader:
#     print(X_batch.shape)
#     print(y_batch.shape)
#     break

In [ ]:
class LogisticRegressionModel(nn.Module):
    """Logistic Regression Model."""

    def __init__(self, input_dim):
        super(LogisticRegressionModel, self).__init__()
        self.linear = nn.Linear(input_dim, 1)

    def forward(self, x):
        out = self.linear(x)
        out = torch.sigmoid(out)
        return out

In [ ]:
torch.manual_seed(123)
model = LogisticRegressionModel(input_dim=2)

In [ ]:
optimizer = torch.optim.SGD(model.parameters(), lr=0.05)
criterion = nn.BCELoss()

In [ ]:
def training_loop(model, train_loader, criterion, optimizer, num_epochs):
    """Training loop for the model."""
    for epoch in range(num_epochs):
        model = model.train()
        for batch_idx, (X_batch, y_batch) in enumerate(train_loader):
            # Forward pass
            outputs = model(X_batch).squeeze()
            loss = criterion(outputs, y_batch)
            # Backward pass and optimization
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        if (epoch + 1) % 10 == 0:
            print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}")

In [ ]:
training_loop(model, train_loader, criterion, optimizer, num_epochs=200)

In [ ]:
# create some test_records and predict
X_test = torch.tensor([[0.2, 0.8], [0.6, 0.4], [0.9, 0.1]], dtype=torch.float32)

model = model.eval()
with torch.no_grad():
    predictions = model(X_test).squeeze()
    predicted_classes = (predictions >= 0.5).int()
    print("Predicted probabilities:", predictions.numpy())
    print("Predicted classes:", predicted_classes.numpy())

In [ ]:
model = model.eval()
with torch.inference_mode():
    predictions = model(X_test).squeeze()
    predicted_classes = (predictions >= 0.5).int()
    print("Predicted probabilities:", predictions.numpy())
    print("Predicted classes:", predicted_classes.numpy())

In [ ]:
def compute_accuracy(model, dataloader):

    model = model.eval()

    correct = 0.0
    total_examples = 0

    for idx, (features, class_labels) in enumerate(dataloader):

        with torch.no_grad():
            probas = model(features)

        pred = torch.where(probas > 0.5, 1, 0)
        lab = class_labels.view(pred.shape).to(pred.dtype)

        compare = lab == pred
        correct += torch.sum(compare)
        total_examples += len(compare)
        acc = correct / total_examples

    return acc.numpy()

In [ ]:
compute_accuracy(model, train_loader)

#### Measuring any metric using torch metrics

In [ ]:
import torchmetrics

In [ ]:
metric = torchmetrics.Accuracy(task="binary")

In [ ]:
def training_loop(model, train_loader, criterion, optimizer, num_epochs):
    """Training loop for the model."""
    for epoch in range(num_epochs):
        model = model.train()
        for batch_idx, (X_batch, y_batch) in enumerate(train_loader):
            # Forward pass
            preds = model(X_batch).squeeze()
            loss = criterion(preds, y_batch)
            # Backward pass and optimization
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            accuracy = metric(preds, y_batch.int())

        # calculate metrics
        accuracy = metric.compute()
        metric.reset()
        if (epoch + 1) % 10 == 0:
            print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}, Accuracy: {accuracy:.4f}")

In [ ]:
model = LogisticRegressionModel(input_dim=2)
optimizer = torch.optim.SGD(model.parameters(), lr=0.05)
criterion = nn.BCELoss()

In [ ]:
training_loop(model, train_loader, criterion, optimizer, num_epochs=200)

In [ ]:
X_test = torch.tensor([[0.2, 0.8], [0.6, 0.4], [0.9, 0.1]], dtype=torch.float32)
y_test = torch.tensor([1, 1, 1], dtype=torch.float32)
model = model.eval()
with torch.inference_mode():
    predictions = model(X_test).squeeze()
    predicted_classes = (predictions >= 0.5).int()
    print("Predicted probabilities:", predictions.numpy())
    print("Predicted classes:", predicted_classes.numpy())
    accuracy = metric(predictions, y_test)
    print("Test Accuracy:", accuracy.numpy())

#################################### END ####################################